# 第17章　债券投资策略与回测

[![在 Colab 打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/albertandking/fixed-income/blob/main/notebooks/ch17_strategies.ipynb) [![在 Binder 打开](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/albertandking/fixed-income/main?labpath=notebooks/ch17_strategies.ipynb)

复现例17.1（骑乘曲线归因）、图17-1、例17.2（绩效指标），并比较主动 vs 被动。


In [ ]:
# 自举单元：在 Colab/Binder 上自动安装本书复用包 fi；本地运行时自动跳过。
import importlib.util, sys, subprocess
if importlib.util.find_spec('fi') is None:
    if 'google.colab' in sys.modules:
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/albertandking/fixed-income.git', '/content/fi-book'], check=False)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '/content/fi-book'], check=False)
    else:
        print('提示：请在仓库根目录执行 `uv sync --extra all` 后再运行本 notebook。')


In [ ]:
import numpy as np
from fi import backtest as bt, data, risk, plotting
from fi.cashflow import make_cashflows
plotting.use_chinese_style()
cv = data.load_sample('cgb_yield_curve')
tenor, yld = cv['tenor'].to_numpy(), (cv['yield_pct']/100).to_numpy()


## 例17.1　骑乘曲线策略归因（买 5 年持有 1 年）


In [ ]:
y_buy = float(np.interp(5, tenor, yld))
y_sell = float(np.interp(4, tenor, yld))
cf, t = make_cashflows(y_buy, 5, 1, 100)
dur = risk.modified_duration(cf, t, y_buy, 1)
a = bt.riding_attribution(y_buy, y_sell, dur, coupon_rate=y_buy, horizon=1)
print(f'5y收益率={y_buy*100:.3f}%  1年后4y收益率={y_sell*100:.3f}%  修正久期={dur:.3f}')
print(f'carry={a["carry"]*100:.3f}%  roll-down={a["rolldown"]*100:.3f}%  总收益={a["total"]*100:.3f}%')
y1 = float(np.interp(1, tenor, yld))
print(f'直接持有1年期({y1*100:.3f}%) -> 骑乘多赚 {(a["total"]-y1)*100:.3f}%')


## 图17-1　不同买入期限的骑乘归因（编程实验 6）


In [ ]:
buy_tenors = [2, 3, 5, 7, 10]
carries, rolls = [], []
for b in buy_tenors:
    yb = float(np.interp(b, tenor, yld)); ys = float(np.interp(b-1, tenor, yld))
    cf, t = make_cashflows(yb, b, 1, 100); d = risk.modified_duration(cf, t, yb, 1)
    r = bt.riding_attribution(yb, ys, d, yb, 1)
    carries.append(r['carry']*100); rolls.append(r['rolldown']*100)
    print(f'买{b}年: carry={r["carry"]*100:.2f}%  roll-down={r["rolldown"]*100:.3f}%  总={r["total"]*100:.2f}%')
fig, ax = plotting.new_axes()
x = [f'{b}Y' for b in buy_tenors]
ax.bar(x, carries, label='carry（票息）')
ax.bar(x, rolls, bottom=carries, label='roll-down（骑乘）')
ax.set_xlabel('买入期限（持有1年）'); ax.set_ylabel('收益贡献 (%)')
ax.set_title('图17-1　骑乘曲线策略收益归因'); ax.legend()
fig.tight_layout()


## 例17.2　绩效指标 + 主动 vs 被动（编程实验 7）


In [ ]:
rng = np.random.default_rng(42)
# 合成两条日收益：被动(buy-hold, 稳健) vs 主动(久期择时, 高波动)
passive = 0.025/252 + rng.normal(0, 0.0006, 252)
active = 0.030/252 + rng.normal(0, 0.0015, 252)
for name, r in [('被动(buy-hold)', passive), ('主动(久期择时)', active)]:
    p = bt.performance(r, 252)
    print(f'{name}: 年化收益={p["ann_return"]*100:.2f}%  波动={p["ann_vol"]*100:.2f}%  夏普={p["sharpe"]:.2f}  最大回撤={p["max_drawdown"]*100:.2f}%')
fig, ax = plotting.new_axes()
ax.plot(bt.nav(passive), label='被动')
ax.plot(bt.nav(active), label='主动')
ax.set_xlabel('交易日'); ax.set_ylabel('累计净值'); ax.set_title('主动 vs 被动 累计净值'); ax.legend()
fig.tight_layout()


---

> 小结：债券收益的大头是 carry + roll-down（稳健、不靠预测），久期择时是主动 alpha；
> 骑乘曲线在上行曲线下可超额获益；绩效用年化收益/波动/夏普/最大回撤评价。
